In [2]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
import pandas as pd
import sqlite3 # Vamos usar para verificar

# --- INÍCIO DA SOLUÇÃO ---

# 1. Pega o diretório do notebook (ex: '.../projeto-vagas-cd/notebooks')
notebook_dir = os.getcwd()

# 2. Sobe um nível para a RAIZ DO PROJETO (ex: '.../projeto-vagas-cd')
#    ESTA É A LINHA QUE CORRIGE O CAMINHO!
PROJECT_ROOT = os.path.dirname(notebook_dir)

# 3. Carregue o arquivo .env a partir da raiz do projeto
env_path = os.path.join(PROJECT_ROOT, '.env')

if not os.path.exists(env_path):
    raise FileNotFoundError(f"Arquivo .env não encontrado. Eu procurei em: {env_path}")

load_dotenv(dotenv_path=env_path)
print(f"Arquivo .env carregado de: {env_path}")

# 4. Pegue a URL RELATIVA que está no .env
relative_db_url = os.getenv("DATABASE_URL") # 'sqlite:///data/vagas.db'

if not relative_db_url:
    raise ValueError("DATABASE_URL não encontrada no arquivo .env")

# 5. Extraia apenas o caminho do arquivo
relative_path = relative_db_url.split('///')[-1] # 'data/vagas.db'

# 6. Crie o CAMINHO ABSOLUTO E COMPLETO
#    Junta a RAÍZ DO PROJETO com o caminho relativo do banco
#    Ex: 'C:/projeto-vagas-cd' + 'data/vagas.db'
absolute_path = os.path.join(PROJECT_ROOT, relative_path)

# 7. Garante que a pasta 'data' na raiz exista
os.makedirs(os.path.dirname(absolute_path), exist_ok=True)

# 8. Crie a URL FINAL para o SQLAlchemy
absolute_path_str = absolute_path.replace('\\', '/')
FINAL_URL = f"sqlite:///{absolute_path_str}"

# --- FIM DA SOLUÇÃO ---

# --- TESTE FINAL ---
print(f"Conectando ao banco em: {FINAL_URL}")

# Verificando as tabelas ANTES de usar o Polars
try:
    conn_sqlite = sqlite3.connect(absolute_path)
    cursor = conn_sqlite.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()
    print("Tabelas encontradas no arquivo:", tables)
    conn_sqlite.close()

    if not tables:
        raise ValueError("O arquivo de banco de dados está vazio ou não tem tabelas.")

except Exception as e:
    print(f"Erro ao inspecionar o banco: {e}")


Arquivo .env carregado de: c:\projeto-vagas-cd\.env
Conectando ao banco em: sqlite:///c:/projeto-vagas-cd/data/vagas.db
Tabelas encontradas no arquivo: [('vagas',)]


In [3]:
df = pd.read_sql("SELECT * FROM vagas", FINAL_URL) # type: ignore
df.head()


,id,companyId,name,description,careerPageName,careerPageLogo,careerPageUrl,publishedDate,applicationDeadline,isRemoteWork,city,state,country,jobUrl,workplaceType,disabilities,skills
0,4806873,37162,Gerente de Inteligência de Mercado - Curitiba/...,Estamos em busca de um Gerente de Inteligência...,Scanntech Brasil,https://attachments.gupy.io/production/compani...,https://scanntechbrasil.gupy.io/eyJzb3VyY2UiOi...,2025-10-28T20:07:21.371Z,None,0,Curitiba,Paraná,Brasil,https://scanntechbrasil.gupy.io/job/eyJqb2JJZC...,on-site,0,[]
1,6464644,444,Operador(a) de Caixa - GUARULHOS(555472),"No GPA, todos(as) são bem-vindos(as).Fazemos q...",GPA,https://attachments.gupy.io/production/compani...,https://gpa.gupy.io/eyJzb3VyY2UiOiJndXB5X3Bvcn...,2025-10-30T15:05:22.456Z,2025-12-30,0,Guarulhos,São Paulo,Brasil,https://gpa.gupy.io/job/eyJqb2JJZCI6NjQ2NDY0NC...,on-site,0,[]
2,7393748,39505,TÉCNICO ELETROELETRONICO I,Quem somosA Companhia Riograndense de Saneamen...,Confidencial,https://career-page-prod.gupy.io/default-creat...,https://pgconfidencial.gupy.io/eyJzb3VyY2UiOiJ...,2025-10-29T20:47:20.691Z,None,0,Lajeado,Rio Grande do Sul,Brasil,https://pgconfidencial.gupy.io/job/eyJqb2JJZCI...,on-site,1,[]
3,7478055,23930,RECEPCIONISTA HOSPITALAR,"A hora de fazer parte de um time comprometido,...",Página de Carreira,https://attachments.gupy.io/production/compani...,https://hapvidandi.gupy.io/eyJzb3VyY2UiOiJndXB...,2025-10-30T17:09:25.851Z,2025-11-06,0,Osasco,São Paulo,Brasil,https://hapvidandi.gupy.io/job/eyJqb2JJZCI6NzQ...,on-site,1,[]
4,7732888,316,ANALISTA DE SUPORTE COMERCIAL l ITAU POWER SHO...,Vice-Presidência B2C&nbsp;Diretoria: Dir Regio...,Vem Pra Vivo!,https://attachments.gupy.io/production/compani...,https://lojasvivo.gupy.io/eyJzb3VyY2UiOiJndXB5...,2025-10-31T12:05:53.993Z,2025-11-30,0,Contagem,Minas Gerais,Brasil,https://lojasvivo.gupy.io/job/eyJqb2JJZCI6Nzcz...,on-site,1,[]


In [4]:
df.drop(columns=['careerPageUrl','careerPageLogo', ], inplace=True)

In [5]:
# Palavras chave para pequisar na descrição do emprego
keywords = ['Python', 'SQL', 'pipeline', 'ETL', 'data engineering', 'data engineer']
filtered_df = df[df['description'].str.contains('|'.join(keywords), case=False)]

# Palavras que não queremos
exclude_keywords = ["Pl",'estágio', 'intern', 'internship', 'sênior', 'pleno', 'senior', 'sr', 'lead', 'coordenador', 'estagiário', 'Jovem Aprendiz']
filtered_df = filtered_df[~filtered_df['name'].str.contains('|'.join(exclude_keywords), case=False)]

# Se não for remoto, entao deve ser do RJ

In [6]:
desired_first_cols = ['name','careerPageName','jobUrl','location', 'publishedDate', 'description', 'companyName', 'applyUrl']

# 2. Crie uma lista das suas colunas desejadas que REALMENTE existem no DataFrame
existing_first_cols = [col for col in desired_first_cols if col in filtered_df.columns]

# 3. Crie uma lista de todas as OUTRAS colunas
other_cols = [col for col in filtered_df.columns if col not in existing_first_cols]

# 4. Combine as duas listas e reordene o DataFrame
df_rearranged = filtered_df[existing_first_cols + other_cols]

print(df_rearranged)

                                                   name        careerPageName  \
64          Especialista de Desenvolvimento de Produtos      Scanntech Brasil   
115   PESSOA CONSULTORA TECNOLOGIA EDUCACIONAL - CUR...            #VempraFTD   
119                                   Developer Analyst                 Topaz   
121   Analista de Gestão de Identidades e Acessos (I...                 Asaas   
133   Técnico de Suporte Júnior - Atendimento ao Cli...                Benner   
...                                                 ...                   ...   
9654                                Analista de Growth           Confidencial   
9710                            Gerente de Fábrica Ágil               Log Lab   
9729  Promotor técnico Pet Society - Porto Alegre e ...  Vetlog Distribuidora   
9731    Promotor técnico Pet Society - Pelotas e região  Vetlog Distribuidora   
9755                    Supervisor de técnico/comercial  Vetlog Distribuidora   

                           

In [12]:
# Vamos testar a lib rake pra extração de palavras chave
from rake_nltk import Rake
rake = Rake()

def extract_keywords(description):
    rake.extract_keywords_from_text(description)
    return ', '.join(rake.get_ranked_phrases()[:5])  # Retorna as 5 principais frases-chave
df_rearranged['keywords'] = df_rearranged['description'].apply(extract_keywords)
df_rearranged.head()

C:\Users\user\AppData\Local\Temp\ipykernel_69432\88229071.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_rearranged['keywords'] = df_rearranged['description'].apply(extract_keywords)


,name,careerPageName,jobUrl,publishedDate,description,id,companyId,applicationDeadline,isRemoteWork,city,state,country,workplaceType,disabilities,skills,keywords
64,Especialista de Desenvolvimento de Produtos,Scanntech Brasil,https://scanntechbrasil.gupy.io/job/eyJqb2JJZC...,2025-10-31T12:17:58.754Z,Buscamos um(a) Especialista de Produtos de int...,9772243,37162,2025-11-29,0,São Paulo,São Paulo,Brasil,on-site,1,[],especialista de produtos de inteligência de me...
115,PESSOA CONSULTORA TECNOLOGIA EDUCACIONAL - CUR...,#VempraFTD,https://vempraftd.gupy.io/job/eyJqb2JJZCI6OTkw...,2025-10-30T21:48:10.472Z,"Há mais de 120 anos, a FTD Educação tem como m...",9904180,53101,2025-11-30,0,Curitiba,Paraná,Brasil,on-site,1,[],vempraftd e faça parte desse time apaixonado p...
119,Developer Analyst,Topaz,https://cobistopaz.gupy.io/job/eyJqb2JJZCI6OTk...,2025-10-28T15:07:58.958Z,¡En Topaz nos une la tecnología y nos conecta ...,9907461,42212,2025-12-02,0,Bogotá,,Colômbia,hybrid,1,[],informações adicionais ¡ ten en cuesta estos b...
121,Analista de Gestão de Identidades e Acessos (I...,Asaas,https://asaas.gupy.io/job/eyJqb2JJZCI6OTkxMzc0...,2025-10-28T18:02:56.719Z,Se você quer iniciar sua trajetória na área de...,9913747,30728,2025-11-28,1,,,Brasil,remote,0,[],que goste de aprender e tenha vontade de cresc...
133,Técnico de Suporte Júnior - Atendimento ao Cli...,Benner,https://vemserbenner.gupy.io/job/eyJqb2JJZCI6O...,2025-10-28T22:45:38.216Z,Temos a missão de facilitar o dia a dia das pe...,9923774,2358,2025-12-31,0,Blumenau,Santa Catarina,Brasil,on-site,1,[],mail e ferramenta sisconweb e demais ferrament...
